# Pre-entrega 3 - Clasificador supervisado con TF-IDF (AG News)
**Data Science III - NLP y Deep Learning**

Pipeline: texto crudo -> preprocesamiento del Modulo 2 (Regex + lowercase + lematizacion spaCy) -> `TfidfVectorizer` -> clasificador -> `classification_report` y matriz de confusion.

**Sin data leakage:** el vectorizador se ajusta (`fit`) solo con train; el test solo se transforma y se usa una unica vez, al final.

> **Antes de ejecutar:** `pip install -r requirements.txt` y `python -m spacy download en_core_web_sm`. Este notebook debe estar en la misma carpeta que `proy_datasc_tp_mlg.py` y la carpeta `data/` (con `ag_news_train.csv` y `ag_news_test.csv`). Ejecuta *Run All* para que queden guardadas las salidas, y entrega el notebook ya ejecutado.

In [13]:
# 1) Librerias y carga de datos (splits provistos por la catedra: nunca se entrena con el test)
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Funciones del Modulo 2 (limpieza Regex, normalizacion y lematizacion) que vive en el script del proyecto
import proy_datasc_tp_mlg as P

train_df, test_df = P.cargar()
clases = sorted(train_df.label.unique())
print("train:", train_df.shape, "| test:", test_df.shape)
print("clases:", clases)
print(train_df.label.value_counts().to_dict())

train: (8000, 2) | test: (2000, 2)
clases: ['Business', 'Sci_Tech', 'Sports', 'World']
{'Sci_Tech': 2000, 'Business': 2000, 'Sports': 2000, 'World': 2000}


## 1. Reutilizacion del preprocesamiento (Modulo 2)
`preprocess_corpus` limpia entidades HTML rotas, tags, URLs y firmas de agencia, pasa a minusculas y lematiza con spaCy. Se aplica **igual** a train y a test (es una transformacion sin parametros aprendidos, por lo que no hay fuga de datos).

In [14]:
# 2) Preprocesamiento del corpus completo
train_df["prep"] = P.preprocess_corpus(train_df.text)
test_df["prep"] = P.preprocess_corpus(test_df.text)
print("Lematizador utilizado:", P._LEMMATIZER)
if "spaCy" not in P._LEMMATIZER:
    print("ATENCION: spaCy no esta instalado; se uso un lematizador de reglas. "
          "Ejecuta `python -m spacy download en_core_web_sm` y vuelve a correr para la entrega.")
train_df[["text", "prep"]].head(3)

Lematizador utilizado: spaCy en_core_web_sm


,text,prep
0,"Red Hat Appoints New CFO. September 2, 2004 --...",red hat appoint new cfo september 2 2004 web h...
1,"Sprint, SBC In Hotspot Roaming Deal. Sprint an...",sprint sbc in hotspot roam deal sprint and sbc...
2,Radio Outage Grounds West Coast Flights. Airpo...,radio outage ground west coast flight airport ...


## 2. Experimentos de vectorizacion y modelo
Se exploran dos hiperparametros del `TfidfVectorizer`:
- **`max_features`** (5.000 / 20.000 / sin limite): limita el vocabulario a los terminos mas frecuentes; reduce dimensionalidad y riesgo de overfitting.
- **`ngram_range`** ((1,1) / (1,2)): unigramas vs. unigramas + bigramas, que capturan contexto local (*new york*, *oil price*).

Ademas: `sublinear_tf=True` (amortigua la frecuencia de terminos muy repetidos), `min_df=2` (descarta terminos que aparecen en un solo documento) y `stop_words="english"` (reduce el tamano de la matriz sin perder senal tematica).

Los 3 clasificadores se comparan sobre una **validacion tomada del train (20%)**; el test no interviene en ninguna decision.

In [15]:
# 3) Validacion interna: se parte SOLO el train
X_tr, X_va, y_tr, y_va = train_test_split(
    train_df.prep, train_df.label, test_size=0.2, stratify=train_df.label, random_state=42)

modelos = {
    "MultinomialNB": lambda: MultinomialNB(alpha=0.5),
    "LogisticRegression": lambda: LogisticRegression(C=10, max_iter=1000),
    "LinearSVC": lambda: LinearSVC(C=0.5, random_state=42),
}

filas = []
for mf in (5000, 20000, None):
    for ng in ((1, 1), (1, 2)):
        vec = TfidfVectorizer(max_features=mf, ngram_range=ng, sublinear_tf=True,
                              min_df=2, stop_words="english")
        A = vec.fit_transform(X_tr)      # fit SOLO sobre el train interno
        B = vec.transform(X_va)          # la validacion solo se transforma
        for nombre, mk in modelos.items():
            clf = mk().fit(A, y_tr)
            f1 = f1_score(y_va, clf.predict(B), average="macro")
            filas.append({"max_features": mf, "ngram_range": ng, "modelo": nombre,
                          "n_features": A.shape[1], "f1_macro_val": round(f1, 4)})

grid = pd.DataFrame(filas).sort_values("f1_macro_val", ascending=False).reset_index(drop=True)
grid

,max_features,ngram_range,modelo,n_features,f1_macro_val
0,NaN,"(1, 2)",MultinomialNB,20507,0.8960
1,20000.0,"(1, 2)",MultinomialNB,20000,0.8954
2,NaN,"(1, 2)",LinearSVC,20507,0.8934
3,20000.0,"(1, 1)",MultinomialNB,8253,0.8929
4,NaN,"(1, 1)",MultinomialNB,8253,0.8929
5,20000.0,"(1, 2)",LinearSVC,20000,0.8921
6,5000.0,"(1, 1)",MultinomialNB,5000,0.8914
7,5000.0,"(1, 2)",MultinomialNB,5000,0.8907
8,20000.0,"(1, 2)",LogisticRegression,20000,0.8891
9,NaN,"(1, 2)",LogisticRegression,20507,0.8879


In [16]:
# 4) Modelo final: mejor configuracion de la grilla, ajustada sobre TODO el train
mejor = grid.iloc[0]
mf = None if pd.isna(mejor.max_features) else int(mejor.max_features)   # pandas convierte None en NaN
vec = TfidfVectorizer(max_features=mf, ngram_range=mejor.ngram_range, sublinear_tf=True,
                      min_df=2, stop_words="english")
t0 = time.time()
Xtr = vec.fit_transform(train_df.prep)            # fit_transform SOLO sobre train
clf = modelos[mejor.modelo]().fit(Xtr, train_df.label)
train_s = time.time() - t0
print(f"Modelo: {mejor.modelo} | max_features={mf} | ngram_range={mejor.ngram_range} "
      f"| features reales={Xtr.shape[1]} | entrenamiento: {train_s:.2f} s")

Modelo: MultinomialNB | max_features=None | ngram_range=(1, 2) | features reales=25400 | entrenamiento: 0.17 s


## 3. Evaluacion final sobre el test (una sola vez)

In [17]:
# 5) Prediccion sobre el test: SOLO transform (sin fit)
Xte = vec.transform(test_df.prep)
pred = clf.predict(Xte)

print(classification_report(test_df.label, pred, labels=clases, digits=4))

cm = confusion_matrix(test_df.label, pred, labels=clases)
plt.figure(figsize=(5.6, 4.6))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(len(clases)), clases, rotation=30)
plt.yticks(range(len(clases)), clases)
for i in range(len(clases)):
    for j in range(len(clases)):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xlabel("Prediccion"); plt.ylabel("Real")
plt.title(f"Matriz de confusion - TF-IDF + {mejor.modelo}")
plt.tight_layout()
plt.savefig("matriz_confusion_m3.png", dpi=150)
plt.show()

              precision    recall  f1-score   support

    Business     0.8395    0.8680    0.8535       500
    Sci_Tech     0.8803    0.8680    0.8741       500
      Sports     0.9490    0.9680    0.9584       500
       World     0.9167    0.8800    0.8980       500

    accuracy                         0.8960      2000
   macro avg     0.8964    0.8960    0.8960      2000
weighted avg     0.8964    0.8960    0.8960      2000



/var/folders/c0/yjbwjs9n2vj_16w1bzm5h63c0000gn/T/ipykernel_45988/201267282.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# 6) Que categorias son mas dificiles (insumo del README)
rep = classification_report(test_df.label, pred, labels=clases, output_dict=True)
f1_por_clase = {c: rep[c]["f1-score"] for c in clases}
dificil = min(f1_por_clase, key=f1_por_clase.get)
facil = max(f1_por_clase, key=f1_por_clase.get)

off = cm.copy(); np.fill_diagonal(off, 0)
pares = sorted(((off[i, j], clases[i], clases[j]) for i in range(len(clases)) for j in range(len(clases)) if i != j), reverse=True)
print("F1 por clase:", {c: round(v, 3) for c, v in f1_por_clase.items()})
print(f"Mas dificil: {dificil} | mas facil: {facil}")
print("Confusiones mas frecuentes (real -> predicho):")
for n, a, b in pares[:3]:
    print(f"  {a} -> {b}: {n} casos")

F1 por clase: {'Business': 0.853, 'Sci_Tech': 0.874, 'Sports': 0.958, 'World': 0.898}
Mas dificil: Business | mas facil: Sports
Confusiones mas frecuentes (real -> predicho):
  Sci_Tech -> Business: 44 casos
  Business -> Sci_Tech: 38 casos
  World -> Business: 34 casos


In [19]:
# 7) Terminos mas influyentes por clase (interpretabilidad del baseline)
W = clf.coef_ if hasattr(clf, "coef_") else clf.feature_log_prob_
vocab = vec.get_feature_names_out()
top_terms = {c: [vocab[j] for j in np.argsort(W[i])[::-1][:8]] for i, c in enumerate(clases)}
for c, t in top_terms.items():
    print(f"{c:9s}: {', '.join(t)}")

Business : say, oil, price, company, new, profit, stock, rise
Sci_Tech : microsoft, new, say, software, internet, company, service, computer
Sports   : win, game, season, team, night, cup, victory, league
World    : say, iraq, kill, president, minister, election, leader, official


## 4. Documentacion
La celda siguiente genera `README.md` (justificacion del modelo, parametros del vectorizador y analisis de categorias dificiles) y `requirements.txt` con las **versiones reales** de tu entorno, a partir de los resultados de esta corrida.

In [20]:
# 8) Genera README.md y requirements.txt con los resultados reales de esta ejecucion
import importlib.metadata as md

def version(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return None

paquetes = ["numpy", "pandas", "scikit-learn", "matplotlib", "spacy"]
reqs = [f"{p}=={version(p)}" for p in paquetes if version(p)]
open("requirements.txt", "w").write("\n".join(reqs) + "\n")

razon = {
    "MultinomialNB": "Naive Bayes multinomial modela conteos/pesos de palabras asumiendo independencia condicional; es muy rapido, "
                     "trabaja directo sobre la matriz esparsa y suele rendir bien en clasificacion tematica de textos cortos.",
    "LogisticRegression": "La regresion logistica es un clasificador lineal con salida probabilistica e interpretable "
                          "(los coeficientes muestran que palabras empujan hacia cada categoria).",
    "LinearSVC": "La SVM lineal maximiza el margen y es muy robusta en alta dimension (muchas mas palabras que documentos), "
                 "por lo que es un estandar para clasificacion de texto clasica.",
}
mejor_por_modelo = grid.groupby("modelo").f1_macro_val.max().sort_values(ascending=False)
tabla_modelos = "\n".join(f"| {m} | {v:.4f} |" for m, v in mejor_por_modelo.items())
tabla_grid = "\n".join(f"| {int(r.max_features) if pd.notna(r.max_features) else 'sin limite'} | {r.ngram_range} | {r.modelo} | {r.n_features} | {r.f1_macro_val:.4f} |"
                        for r in grid.head(8).itertuples())
macro = rep["macro avg"]
tabla_clases = "\n".join(f"| {c} | {rep[c]['precision']:.4f} | {rep[c]['recall']:.4f} | {rep[c]['f1-score']:.4f} |" for c in clases)
confs = "\n".join(f"- **{a} -> {b}**: {n} documentos" for n, a, b in pares[:3])
terminos = "\n".join(f"- **{c}**: {', '.join(t[:6])}" for c, t in top_terms.items())

readme = f'''# Pre-entrega 3 - Clasificador supervisado con TF-IDF (AG News)

Baseline clasico para el proyecto de Data Science III: texto crudo -> preprocesamiento (Modulo 2) -> TF-IDF -> clasificador.

## Contenido
- `M3_clasificador_tfidf.ipynb`: pipeline completo y comentado.
- `proy_datasc_tp_mlg.py`: funciones de preprocesamiento del Modulo 2 (limpieza Regex, normalizacion y lematizacion).
- `data/`: `ag_news_train.csv` (8.000) y `ag_news_test.csv` (2.000).
- `requirements.txt`: versiones usadas.

## Datos y ausencia de data leakage
Se usan los splits provistos. El `TfidfVectorizer` se ajusta (`fit_transform`) **solo con train** y se aplica `transform` al test. La eleccion de
hiperparametros y modelo se hizo en una validacion del 20% tomada del train; el test se uso una unica vez al final.
Preprocesamiento: limpieza Regex (entidades HTML, tags, URLs, firmas de agencia), minusculas y lematizacion con **{P._LEMMATIZER}**.

## Parametros del vectorizador (elegidos por validacion)
| Parametro | Valor | Motivo |
|---|---|---|
| max_features | {mf if mf else 'sin limite'} | limita el vocabulario a los terminos mas frecuentes y reduce overfitting y memoria |
| ngram_range | {mejor.ngram_range} | {'incluye bigramas para capturar contexto local (p. ej. "new york", "oil price")' if mejor.ngram_range == (1, 2) else 'solo unigramas'} |
| sublinear_tf | True | amortigua terminos que se repiten muchas veces en un documento |
| min_df | 2 | descarta terminos que aparecen en un unico documento (ruido y errores tipograficos) |
| stop_words | english | reduce el tamano de la matriz sin perder senal tematica |

Vocabulario resultante: {Xtr.shape[1]:,} terminos. Mejores configuraciones en validacion (F1 macro):

| max_features | ngram_range | modelo | n_features | F1 macro val. |
|---|---|---|---|---|
{tabla_grid}

## Justificacion del modelo
Se compararon Naive Bayes, Regresion Logistica y SVM lineal con la misma grilla de vectorizacion. Mejor F1 macro de validacion por modelo:

| Modelo | F1 macro (val.) |
|---|---|
{tabla_modelos}

Se eligio **{mejor.modelo}** por obtener el mejor F1 macro en validacion. {razon[mejor.modelo]}
Entrenamiento sobre el train completo: {train_s:.2f} s en CPU.

## Resultados sobre el test
F1 macro = **{macro["f1-score"]:.4f}** | Accuracy = **{rep["accuracy"]:.4f}** (las 4 clases estan balanceadas: 500 documentos por clase en test, por lo que macro y weighted coinciden).

| Clase | Precision | Recall | F1 |
|---|---|---|---|
{tabla_clases}

![Matriz de confusion](matriz_confusion_m3.png)

## Analisis preliminar: categorias mas dificiles
Segun la matriz de confusion, la categoria mas dificil es **{dificil}** (F1 = {f1_por_clase[dificil]:.3f}) y la mas facil es **{facil}** (F1 = {f1_por_clase[facil]:.3f}).
Confusiones mas frecuentes (real -> predicho):

{confs}

Terminos mas influyentes por categoria:

{terminos}

La confusion mas frecuente es **{pares[0][1]} -> {pares[0][2]}** ({pares[0][0]} documentos). Es esperable en un modelo de bolsa de palabras: TF-IDF no captura el contexto ni el orden de las palabras,
por lo que las categorias que comparten vocabulario se separan peor. Esa limitacion es la que se busca mejorar con el Transformer + LoRA del Modulo 4.

## Como reproducir
```
pip install -r requirements.txt
python -m spacy download en_core_web_sm
jupyter notebook M3_clasificador_tfidf.ipynb   # Run All
```
'''
open("README.md", "w", encoding="utf-8").write(readme)
print("Generados: README.md, requirements.txt, matriz_confusion_m3.png")
print(open("requirements.txt").read())

Generados: README.md, requirements.txt, matriz_confusion_m3.png
numpy==2.5.0
pandas==3.0.3
scikit-learn==1.9.0
matplotlib==3.11.0
spacy==3.8.18

